# AutoGluon.TimeSeries V1.4 Cheatsheet

This notebook contains the key code snippets from the AutoGluon Time Series cheatsheet.

In [1]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

/home/sergio/code/autogluon/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Preparing Data

AutoGluon can generate forecasts for datasets consisting of **multiple univariates** time series. Here we use the M4 Competition Daily dataset to demonstrate how to do forecasting with AutoGluon. 

The data typically requires two parts: **raw data** (time series) and **static features** (metadata).

In [2]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

# 1. Load the raw time series data (time series values)
# NOTE: You would replace 'm4_daily.csv' with your own dataset path.
raw_data = pd.read_csv("data/201401GxSEN.csv",sep=";",low_memory=False)
print("Raw Data Head:")
print(raw_data.columns)



Raw Data Head:
Index(['Date', 'Year', 'Month', 'Day', 'Hour', 'Central', 'Unidad',
       'Componente', 'Generacion_MWh', 'Tecnologia', 'Clasificacion',
       'Codigo Central', 'Subsistema'],
      dtype='object')


In [3]:
raw_data["Hour"] = (raw_data['Hour'].astype(int) - 1).apply(lambda x: f"{x:02d}:00")
raw_data["Timestamp"] = pd.to_datetime(raw_data["Date"]+' '+raw_data["Hour"], format="%d/%m/%Y %H:%M")

In [4]:
raw_data["Generacion_MWh"]=raw_data["Generacion_MWh"].str.replace(',','.').astype("float")
raw_data["Codigo Central"]=raw_data["Codigo Central"].astype("category")
raw_data["Tecnologia"]=raw_data["Tecnologia"].astype("category")
raw_data["Clasificacion"]=raw_data["Clasificacion"].astype("category")

In [17]:
raw_data["Tecnologia"].unique()

['Mini Hidráulica de Pasada', 'Hidráulica de Pasada', 'Hidráulica de Embalse', 'Gas Natural', 'Petróleo Diesel', 'Carbón', 'Biomasa', 'Eólica', 'Solar Fotovoltaica', 'Cogeneración']
Categories (10, object): ['Biomasa', 'Carbón', 'Cogeneración', 'Eólica', ..., 'Hidráulica de Pasada', 'Mini Hidráulica de Pasada', 'Petróleo Diesel', 'Solar Fotovoltaica']

In [18]:
from unidecode import unidecode

# Function to remove accents using unidecode
def remove_accents(text):
    if isinstance(text, str):
        return unidecode(text)
    else:
        return text

raw_data["Tecnologia"]=raw_data["Tecnologia"].cat.rename_categories(lambda x: x.upper().replace(' ','_'))
raw_data["Clasificacion"]=raw_data["Clasificacion"].cat.rename_categories(lambda x: x.upper().replace(' ','_'))  
raw_data["Codigo Central"]=raw_data["Codigo Central"].cat.rename_categories(lambda x: x.upper().replace(' ','_'))
raw_data["Codigo Central"]=raw_data["Codigo Central"].cat.rename_categories(remove_accents)
raw_data["Codigo Central"]=raw_data["Codigo Central"].cat.rename_categories(lambda x: x.replace('-','_').replace('.','_'),)
raw_data["Codigo Central"]=raw_data["Codigo Central"].cat.rename_categories(lambda x: 'C_' + x)
raw_data['Tecnologia']=raw_data['Tecnologia'].apply(remove_accents)

In [19]:
raw_data['Tecnologia'].unique()

['MINI_HIDRAULICA_DE_PASADA', 'HIDRAULICA_DE_PASADA', 'HIDRAULICA_DE_EMBALSE', 'GAS_NATURAL', 'PETROLEO_DIESEL', 'CARBON', 'BIOMASA', 'EOLICA', 'SOLAR_FOTOVOLTAICA', 'COGENERACION']
Categories (10, object): ['BIOMASA', 'CARBON', 'COGENERACION', 'EOLICA', ..., 'HIDRAULICA_DE_PASADA', 'MINI_HIDRAULICA_DE_PASADA', 'PETROLEO_DIESEL', 'SOLAR_FOTOVOLTAICA']

In [20]:
filtered_df = raw_data[raw_data['Tecnologia'].isin(['MINI_HIDRAULICA_DE_PASADA', 'HIDRAULICA_DE_PASADA', 'HIDRAULICA_DE_EMBALSE','EOLICA', 'SOLAR_FOTOVOLTAICA'])]

In [ ]:
filtered_df["Tecnologia"] = filtered_df["Tecnologia"].cat.remove_unused_categories()

/tmp/ipykernel_101958/1591513818.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["Tecnologia"] = filtered_df["Tecnologia"].cat.remove_unused_categories()


In [38]:
filtered_df.groupby("Tecnologia",observed=True)["Generacion_MWh"].sum().sort_values(ascending=False)

Tecnologia
HIDRAULICA_DE_PASADA         1009564.97
HIDRAULICA_DE_EMBALSE         784242.00
MINI_HIDRAULICA_DE_PASADA     128471.98
EOLICA                         57156.60
SOLAR_FOTOVOLTAICA              4470.15
Name: Generacion_MWh, dtype: float64

In [39]:
generacion=filtered_df[["Codigo Central","Timestamp","Generacion_MWh"]]

In [40]:
generacion_static=filtered_df[["Codigo Central","Tecnologia","Clasificacion"]].drop_duplicates().reset_index(drop=True)

In [41]:
print("Raw Data Head:")
print(generacion.head())
print("\nStatic Features Head:")
print(generacion_static.head())

Raw Data Head:
  Codigo Central           Timestamp  Generacion_MWh
0   C_LOS_MOLLES 2014-01-01 00:00:00             0.0
1   C_LOS_MOLLES 2014-01-01 01:00:00             0.0
2   C_LOS_MOLLES 2014-01-01 02:00:00             0.0
3   C_LOS_MOLLES 2014-01-01 03:00:00             0.0
4   C_LOS_MOLLES 2014-01-01 04:00:00             0.0

Static Features Head:
  Codigo Central                 Tecnologia Clasificacion
0   C_LOS_MOLLES  MINI_HIDRAULICA_DE_PASADA          ERNC
1  C_SAUCE_ANDES  MINI_HIDRAULICA_DE_PASADA          ERNC
2    C_SAUZALITO  MINI_HIDRAULICA_DE_PASADA          ERNC
3    C_ACONCAGUA       HIDRAULICA_DE_PASADA  CONVENCIONAL
4       C_SAUZAL       HIDRAULICA_DE_PASADA  CONVENCIONAL


### Convert Raw Data into a TimeSeriesDataFrame

Convert your raw data into the format required by AutoGluon: `TimeSeriesDataFrame`.

In [42]:
# from autogluon.timeseries import TimeSeriesDataFrame # Already imported above
train_data = TimeSeriesDataFrame(
    generacion,
    id_column="Codigo Central",
    timestamp_column="Timestamp",
    static_features=generacion_static,  # Optional metadata/covariates
)

print("Processed Training Data:")
print(train_data.head(5))

Processed Training Data:
                                  Generacion_MWh
item_id      timestamp                          
C_LOS_MOLLES 2014-01-01 00:00:00             0.0
             2014-01-01 01:00:00             0.0
             2014-01-01 02:00:00             0.0
             2014-01-01 03:00:00             0.0
             2014-01-01 04:00:00             0.0


In [43]:
train_data.static_features.dtypes

Tecnologia       category
Clasificacion    category
dtype: object

In [45]:
train_data.groupby("item_id",observed=True).count()

,Generacion_MWh
item_id,
C_ABANICO,744
C_ACONCAGUA,1488
C_ALFALFAL,744
C_ALLIPEN,744
C_ANGOSTURA,720
...,...
C_TAMBO_REAL,744
C_TRUENO,744
C_TRUFUL_TRUFUL,744


In [47]:
train_data.convert_frequency(freq="H")

/home/sergio/code/autogluon/.venv/lib/python3.12/site-packages/autogluon/timeseries/dataset/ts_dataframe.py:1071: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset = pd.tseries.frequencies.to_offset(freq)
/home/sergio/code/autogluon/.venv/lib/python3.12/site-packages/autogluon/timeseries/dataset/ts_dataframe.py:1099: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  chunks = split_into_chunks(df.groupby(level=ITEMID, sort=False), chunk_size)


Generacion_MWh
item_id      timestamp                          
C_LOS_MOLLES 2014-01-01 00:00:00             0.0
             2014-01-01 01:00:00             0.0
             2014-01-01 02:00:00             0.0
             2014-01-01 03:00:00             0.0
             2014-01-01 04:00:00             0.0
...                                          ...
C_HUAYCA1    2014-01-31 19:00:00             0.0
             2014-01-31 20:00:00             0.0
             2014-01-31 21:00:00             0.0
             2014-01-31 22:00:00             0.0
             2014-01-31 23:00:00             0.0

[80664 rows x 1 columns]

## Training

Train models to forecast the values in the column `target` $30$ steps into the future.

In [49]:
# from autogluon.timeseries import TimeSeriesPredictor # Already imported above

predictor = TimeSeriesPredictor(
    target="Generacion_MWh",
    prediction_length=24*7, # The number of time steps into the future to forecast
    freq="H",
    # Optional: Additional covariates that are known in the future
    # known_covariates_names=['weekday', 'month'],
).fit(
    train_data,
    # Presets control the model quality and training time
    presets="medium_quality",
    # Other options: tuning metric, time limit, hyperparamter adjustment, etc.
    # eval_metric="MAPE",
    # time_limit=600,
    # verbosity=2
)

/home/sergio/code/autogluon/.venv/lib/python3.12/site-packages/autogluon/timeseries/predictor.py:198: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset = pd.tseries.frequencies.to_offset(self.freq)
Frequency 'H' stored as 'h'
Beginning AutoGluon training...
AutoGluon will save models to '/home/sergio/code/autogluon/AutogluonModels/ag-20251003_184458'
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #32~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Sep  2 14:21:04 UTC 2
CPU Count:          12
GPU Count:          1
Memory Avail:       53.26 GB / 62.70 GB (85.0%)
Disk Space Avail:   109.94 GB / 217.97 GB (50.4%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'h',
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,


## Predicting

Forecast `prediction_length` steps into the future starting from the end of each time series in `train_data`.

In [50]:
predictions = predictor.predict(
    train_data,
    # If you used known_covariates_names during training, pass them here:
    # known_covariates=known_covariates,
)

print("Forecasted Predictions Head:")
print(predictions.head())

# AutoGluon generates probabilistic forecasts that include:
# - mean_forecast - expected value of the time series
# - quantile_forecast - range of possible outcomes

# Predict on a new, unseen dataset
# predictions_test = predictor.predict_test_data(
#     test_data,
#     model_names=['DeepAR', 'ETS'], # Optional: specify which models to use
# )

data with frequency 'IRREG' has been resampled to frequency 'h'.
/home/sergio/code/autogluon/.venv/lib/python3.12/site-packages/autogluon/timeseries/dataset/ts_dataframe.py:1099: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  chunks = split_into_chunks(df.groupby(level=ITEMID, sort=False), chunk_size)
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


Forecasted Predictions Head:
                                      mean       0.1       0.2       0.3  \
item_id      timestamp                                                     
C_LOS_MOLLES 2014-02-01 00:00:00  3.781561  1.778555  2.483409  2.947473   
             2014-02-01 01:00:00  1.042002 -0.136371  0.226928  0.462224   
             2014-02-01 02:00:00  0.074573 -0.564509 -0.346158 -0.252076   
             2014-02-01 03:00:00  0.012646 -0.451073 -0.273299 -0.212166   
             2014-02-01 04:00:00 -0.117104 -0.502410 -0.367812 -0.328635   

                                       0.4       0.5       0.6       0.7  \
item_id      timestamp                                                     
C_LOS_MOLLES 2014-02-01 00:00:00  3.391076  3.781561  4.242594  4.740453   
             2014-02-01 01:00:00  0.749620  1.042002  1.318063  1.645284   
             2014-02-01 02:00:00 -0.087450  0.074573  0.230812  0.427319   
             2014-02-01 03:00:00 -0.096053  0.012646  0.09

## Model Understanding

Understand the contribution of each model.

In [51]:
leaderboard = predictor.leaderboard()
print(leaderboard)

                       model  score_val  pred_time_val  fit_time_marginal  \
0           WeightedEnsemble  -0.170099       8.793478           3.192115   
1        Chronos[bolt_small]  -0.172235       3.097360           3.094895   
2              DirectTabular  -0.207729       0.722841          11.047742   
3  TemporalFusionTransformer  -0.215012       0.153085         119.337934   
4              SeasonalNaive  -0.314824       0.228228           0.121201   
5           RecursiveTabular  -0.462739       1.388316           1.464793   
6                      Naive  -0.547916       2.371716           0.132977   
7                      Theta  -0.685745       0.540573           0.094387   
8                        ETS  -0.745139       4.154716           0.093170   

   fit_order  
0          9  
1          7  
2          4  
3          8  
4          2  
5          3  
6          1  
7          6  
8          5  
